In [15]:
from typing import List
import numpy as np
import os
from dotenv import load_dotenv

from qiskit import QuantumCircuit, generate_preset_pass_manager, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit_aer import AerSimulator
import qiskit.quantum_info as qi 

simulated = 'FakeBrisbane'  # Change this to simulate other QPUs
dynamic_fake_provider = __import__('qiskit_ibm_runtime.fake_provider', fromlist=[simulated])
fake = getattr(dynamic_fake_provider, simulated)

load_dotenv()

False

In [16]:

TYPE = 'simulated_noise' # simulates a qpu instead of running on ibm
# Length of bitstream produced will be SHOTS * (NUM_QUBITS / PHYSICAL_PER_LOGICAL)
CHANNEL = 'local' # Channel to use for the runtime service. ibm_quantum or local
SHOTS = 10000 # How many times to run the circuit.
NUM_QUBITS = 9 # How many physical qubits to use. Must be a multiple of PHYSICAL_PER_LOGICAL.
PHYSICAL_PER_LOGICAL = 3 # How many physical qubits per logical qubit. Must be an odd number.
if TYPE == 'simulated_noise':
    FILENAME = f'100_k{simulated}_sim' # Filename to save the bitstream to.
else:
    FILENAME = 'bitstream'
assert NUM_QUBITS % PHYSICAL_PER_LOGICAL == 0
assert PHYSICAL_PER_LOGICAL % 2 == 1
assert (SHOTS * (NUM_QUBITS / PHYSICAL_PER_LOGICAL)) % 8 == 0 # Must be a multiple of 8 for the bitstream to be byte-aligned

In [17]:
# Defining Circuit
circ = QuantumCircuit(NUM_QUBITS)
circ.h(range(NUM_QUBITS))
circ.measure_all()

In [18]:
if TYPE == 'simulated_noise':
    device_backend = fake()
    sim_fake = AerSimulator.from_backend(device_backend)
    
    # Transpile for noisy gates
    circ = transpile(circ, sim_fake)
    
    # Run the circuit
    result = sim_fake.run(circ, shots=SHOTS).result()
    counts = result.get_counts(0)
    print(counts)
    
    # Convert counts to bitstrings
    bitstrings = list(counts.keys())
    
    # Perform a logical qubit majority vote
    bitstream = []
    for bitstring in bitstrings:
        bitstring = np.array([int(c) for c in bitstring]).reshape(-1, PHYSICAL_PER_LOGICAL)
        majority_votes = (np.mean(bitstring, axis=1) > 0.5).astype(int).tolist()
        bitstream.extend(majority_votes)
    
    bitstring = [str(bit) for bit in bitstream]
    output = [bitstring[i:i+8] for i in range(0, len(bitstring), 8)]
    ba = bytearray(int(''.join(byte), 2) for byte in output)
    bs = bytes(ba)
    
    with open(f"./{FILENAME}.bin", 'wb') as f:
        f.write(bs)
else:
    service = QiskitRuntimeService(channel=CHANNEL, token=os.getenv('IBMQ_API_TOKEN'))
    if CHANNEL == 'ibm_quantum':
        backend = service.least_busy(simulator=False, operational=True)
    else:
        backend = service.least_busy()
    
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    isa_circ = pm.run(circ)
    
    # Initialise Sampler
    sampler = SamplerV2(mode=backend)
    sampler.options.default_shots = SHOTS
    
    # Run the circuit
    job = sampler.run([isa_circ])
    print(f">>> Job ID: {job.job_id()}")
    result = job.result()
    bitstrings = np.array(result[0].data.meas.get_bitstrings())
    
    # Perform a logical qubit majority vote
    bitstream = []
    for bitstring in bitstrings:
        bitstring = np.array([int(c) for c in bitstring]).reshape(-1, PHYSICAL_PER_LOGICAL)
        majority_votes = (np.mean(bitstring, axis=1) > 0.5).astype(int).tolist()
        bitstream.extend(majority_votes)

{'111101001': 19, '011000101': 15, '100101101': 20, '101010000': 19, '101000111': 14, '100001001': 13, '001001010': 13, '101100111': 18, '111110101': 16, '010110100': 13, '011101110': 13, '110100100': 16, '010101001': 18, '000110101': 9, '111101000': 17, '100000111': 16, '110011011': 17, '110110001': 15, '101001101': 18, '110010100': 10, '110110100': 21, '011011111': 22, '100110011': 11, '111101010': 23, '000010101': 28, '001011001': 13, '001110011': 16, '100000101': 22, '001101011': 15, '110000001': 20, '010010100': 12, '110110011': 16, '010001000': 15, '110110000': 17, '111000111': 7, '000011101': 18, '111001011': 19, '100111010': 12, '011000000': 22, '101111001': 15, '100100100': 21, '100001111': 22, '010010110': 16, '110100001': 21, '011011011': 14, '001011110': 18, '110111011': 13, '111101101': 25, '111100111': 18, '110000111': 17, '110010001': 17, '000111001': 19, '000010110': 19, '011001010': 17, '100000011': 17, '010001001': 25, '011110000': 21, '010000100': 14, '100110001': 17

In [ ]:
bitstring = [str(bit) for bit in bitstream]
output = [bitstring[i:i+8] for i in range(0, len(bitstring), 8)]
ba = bytearray(int(''.join(byte), 2) for byte in output)
bs = bytes(ba)

with open(f"./{FILENAME}.bin", 'wb') as f:
   f.write(bs)